# Proto-RSet: edit a trained ProtoPNet

Run top to bottom in a GPU session with the **protopnext-rset** kernel (not `uvvenv`, which is proto-rset's environment).

1. Edit the **Settings** cell: `CHECKPOINT` is the trained model.
2. Run through **Look at the prototypes**, then put the prototypes you want gone in `REMOVE`.
3. Run the rest: accuracy and activation precision before vs. after, the two figures, and the edited model.

Point `CHECKPOINT` at a file training is no longer overwriting (a finished run, or a copy): the notebook loads it more than once. Everything is written under `OUT`. The data folder is only read. To try a different edit, rerun from **Fit the Proto-RSet**.

In [ ]:
from pathlib import Path

import torch

# ---- Settings: the only cell you need to edit ----
CHECKPOINT = Path("/work/pi_alina_barnett_uri_edu/shirish_khanal_uri_edu/protopnext-rset/runs/vgg16_cosine_finer_c0.000001_csTrue_p15_o0.01_ff0.0952/runs/checkpoints/1816498/1816498_best.pth")
DATASET = "iaiabl_finer"     # dataset name used for training
DATA_DIR = Path("/work/pi_alina_barnett_uri_edu/shirish_khanal_uri_edu/proto-rset-iaiabl/Lo1136i_paper_finer_png")  # read only
FINE_ANNOTATION = True       # trained with fa_coef set, so the data has lesion masks: show activation precision
AP_SPLITS = {                # where activation precision is reported (each split needs a <split>_masks folder)
    "lesion (box masks, test)": "test_DONOTTOUCH",
    "fine (finer masks, test_finer)": "test_finer",
}
OUT =CHECKPOINT.parent / "protorset"  # everything this notebook writes
RSET_ARGS = {"rashomon_bound_multiplier": 1.1, "lam": 0.0001, "max_iter": 5000, "lr_for_opt": 1.0}
BATCH_SIZES = {"train": 20, "project": 8, "val": 50}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE)

## Data

In [ ]:
from protopnet.datasets import training_dataloaders

loaders = training_dataloaders(DATASET, data_path=DATA_DIR, batch_sizes=BATCH_SIZES)
CLASSES = [loaders.class_name_ref_dict[i] for i in range(loaders.num_classes)]
VAL = loaders.val_loader.dataset
MEAN = torch.as_tensor(loaders.train_distribution.mean)
STD = torch.as_tensor(loaders.train_distribution.std)
AP_LOADERS = {label: training_dataloaders(DATASET, data_path=DATA_DIR, val_dir=split, batch_sizes=BATCH_SIZES).val_loader
              for label, split in AP_SPLITS.items()} if FINE_ANNOTATION else {}
print("classes:", CLASSES)
print(f"projection images: {len(loaders.project_loader.dataset)} | validation images: {len(VAL)}")
for label, loader in AP_LOADERS.items():
    print(f"{label}: {len(loader.dataset)} images")

## The trained model, before any edit

In [ ]:
from protopnet.metrics import activation_precision_score


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for batch in loader:
        logits = model(batch["img"].to(DEVICE))["logits"]
        correct += (logits.argmax(1).cpu() == batch["target"]).sum().item()
        total += len(batch["target"])
    return correct / total


def show_activation_precision(model, title):
    # overall and per class, for each split in AP_SPLITS
    print(f"=== {title} ===")
    for label, loader in AP_LOADERS.items():
        overall = activation_precision_score(model, loader, DEVICE)
        per_class = activation_precision_score(model, loader, DEVICE, by="class")
        print(f"  {label:32s} overall {overall:.3f}")
        print("     per class:   " + " | ".join(f"{name} {mean:.3f}" for name, (mean, _) in per_class.items()))


trained = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False).eval()
print(f"trained model: {trained.prototype_layer.num_prototypes} prototypes, "
      f"validation accuracy {accuracy(trained, loaders.val_loader):.4f}")
if FINE_ANNOTATION:
    show_activation_precision(trained, "trained model")

## Fit the Proto-RSet

This loads the model, removes duplicate prototypes, computes every prototype's similarity on the projection and validation images, and fits the set of last layers within `rashomon_bound_multiplier` × the best loss. **From here on, prototype indices refer to this model.**

In [ ]:
import copy

from rashomon_sets.protorset_factory import ProtoRSetFactory

rset_factory = ProtoRSetFactory(
    split_dataloaders=loaders,
    initial_protopnet_path=CHECKPOINT,
    rashomon_set_args=RSET_ARGS,
    device=DEVICE,
    analysis_save_dir=OUT,
)
BEFORE = copy.deepcopy(rset_factory.produce_protopnet_object())  # the Rashomon set's optimal model, before any edit
PROTO_DIR = rset_factory.vis_save_loc / "prototypes"
PROTO_CLASS = BEFORE.prototype_layer.prototype_class_identity.argmax(dim=1).tolist()
print(f"Proto-RSet optimal model: validation accuracy {rset_factory._best_val_acc().item():.4f}")
print(f"{BEFORE.prototype_layer.num_prototypes} prototypes after removing duplicates")
if FINE_ANNOTATION:
    show_activation_precision(BEFORE, "Proto-RSet optimal model")

## Look at the prototypes

Each prototype's class, last-layer weight and activation precision on the validation images, whose masks are lesion boxes (the share of its strongest activation that falls on the lesion; low means it looks outside the lesion). Then each class's prototypes: the source patch (box) and where it activates.

In [ ]:
import pandas as pd

weights = BEFORE.prototype_prediction_head.class_connection_layer.weight.detach().cpu()
per_split = {label: activation_precision_score(BEFORE, loader, DEVICE, by="prototype")
             for label, loader in AP_LOADERS.items()}  # empty when FINE_ANNOTATION is off
TABLE = pd.DataFrame([
    {"prototype": j, "class": CLASSES[c], "weight": round(float(weights[c, j]), 3),
     **{f"AP {label}": round(per[j][2], 3) if j in per else None for label, per in per_split.items()}}
    for j, c in enumerate(PROTO_CLASS)
]).set_index("prototype")
TABLE


In [ ]:
import matplotlib.pyplot as plt

from protopnet.visualization import save_prototype_images_to_file
from protopnet.visualization import plot_prototype_grid

# render once (slow); the images are the same for BEFORE and AFTER
save_prototype_images_to_file(BEFORE, loaders.project_loader, rset_factory.vis_save_loc, loaders.image_size,
                              MEAN, STD, device=DEVICE)

for fig in plot_prototype_grid(BEFORE, PROTO_DIR, AP_LOADERS, DEVICE, class_names=CLASSES):
    plt.show()


In [ ]:
from protopnet.visualization import plot_image_decisions

OUTLINE_SPLIT = "fine (finer masks, test_finer)" 
if FINE_ANNOTATION:
    fig, DECISIONS = plot_image_decisions(BEFORE, AP_LOADERS[OUTLINE_SPLIT].dataset, PROTO_DIR, MEAN, STD, DEVICE,
                                          top_k=3,         
                                          max_images=10, 
                                          class_names=CLASSES, split=OUTLINE_SPLIT)
    plt.show()


In [ ]:
from IPython.display import display
from PIL import Image

CANDIDATES = [16]  # prototypes to show
SCALE = 3  # enlargement for viewing (Lanczos); the saved images are at the model's input size

for j in CANDIDATES:
    gray = Image.open(PROTO_DIR / f"proto_{j}_original.png").convert("L").convert("RGB")  # black and white
    heat = Image.open(PROTO_DIR / f"proto_{j}_overlayheatmap.png").convert("RGB")
    w, h = gray.size
    both = Image.new("RGB", (2 * w + 8, h), "white")  # 8-pixel white gap between the two
    both.paste(gray, (0, 0))
    both.paste(heat, (w + 8, 0))
    both = both.resize((both.width * SCALE, both.height * SCALE), Image.LANCZOS)
    print(f"prototype {j}: source image | activation heatmap")
    display(both)
    both.save(OUT / f"prototype_{j}_gray_heatmap.png")  # same quality as shown


## Edit

Put the prototypes you want removed in `REMOVE`. Each removal is kept only if a model in the Rashomon set can do without that prototype. `REQUIRED` optionally asks for a minimum weight on prototypes you want kept strong.

In [ ]:
REMOVE = [0,5,12,13,16]    # prototype indices to remove, e.g. [3, 17]
REQUIRED = {1:3.0}  # optional: {prototype index: minimum last-layer weight}, e.g. {5: 3.0}

REMOVED = {j: rset_factory.require_to_avoid_prototype(j) for j in REMOVE}
for j, ok in REMOVED.items():
    print(f"remove prototype {j}: {'removed' if ok else 'not possible within the Rashomon set'}")

requirements = list(REQUIRED.items())
if requirements:
    try:
        feasible = rset_factory.rset._sample_model_with_constraints(requirements) is not None
        if not feasible:
            print(f"the requirements {REQUIRED} cannot all hold within the Rashomon set; ignoring them")
    except Exception as error:  
        print(f"the solver could not check {REQUIRED} ({type(error).__name__}: {error}); ignoring them")
        feasible = False
    if not feasible:
        requirements = []
AFTER = (rset_factory.produce_protopnet_object_with_requirements(requirements) if requirements
         else copy.deepcopy(rset_factory.produce_protopnet_object()))

## Before vs. after the edit

In [ ]:
print(f"validation accuracy: {accuracy(BEFORE, loaders.val_loader):.4f} -> {accuracy(AFTER, loaders.val_loader):.4f}")
if FINE_ANNOTATION:
    show_activation_precision(BEFORE, "before edits")
    show_activation_precision(AFTER, "after edits")

## Figures

In [ ]:
from rashomon_sets.visualization import plot_prototype_importance

ROWS = sorted(set(REMOVE) | set(REQUIRED))  # add other prototype indices to show them too
if ROWS:
    plot_prototype_importance(BEFORE, AFTER, PROTO_DIR, prototypes=ROWS, required={j: None for j, _ in requirements},
                              class_names=CLASSES, save_path=OUT / "prototype_importance.png")
    plt.show()
else:
    print("nothing edited yet: fill in REMOVE or REQUIRED")

In [ ]:
from rashomon_sets.visualization import pam_gains, plot_before_after, print_largest_pam_gains

TOP_IMAGES = 3  # validation images with the largest gain in activation precision
if FINE_ANNOTATION:
    gains = pam_gains(BEFORE, AFTER, VAL, DEVICE)
    if not gains:
        print("no annotated validation image of an edited class")
    print_largest_pam_gains(gains, VAL, top_n=TOP_IMAGES, class_names=CLASSES)
    for _, i, _, _ in gains[:TOP_IMAGES]:
        fig = plot_before_after(BEFORE, AFTER, VAL, i, MEAN, STD, DEVICE, class_names=CLASSES, split="validation")
        fig.savefig(OUT / f"activation_before_after_image{i}.png", dpi=150, bbox_inches="tight")
        plt.show()

## Save

In [ ]:
import json

torch.save(AFTER, OUT / "edited_model.pth")
(OUT / "edits.json").write_text(json.dumps(
    {"checkpoint": str(CHECKPOINT), "rset_args": RSET_ARGS, "removed": REMOVED,
     "required_requested": REQUIRED, "required_applied": dict(requirements)}, indent=2))
print("saved", OUT / "edited_model.pth", "and", OUT / "edits.json")